In [ ]:


import numpy as np # library for numerical operations
import pandas as pd # library for data manipulation and analysis
from sklearn import tree # module for decision tree algorithms
from sklearn.model_selection import train_test_split # module for splitting data into training and testing sets
from sklearn.metrics import (
    accuracy_score,
    precision_score, 
    recall_score, 
    f1_score, 
) # function for calculating accuracy score
from sklearn.metrics import confusion_matrix # function for calculating confusion matrix
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold



In [ ]:
%pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 4.2/101.7 MB 20.1 MB/s eta 0:00:05
   --- ------------------------------------ 10.0/101.7 MB 24.0 MB/s eta 0:00:04
   ------ --------------------------------- 16.0/101.7 MB 25.5 MB/s eta 0:00:04
   -------- ------------------------------- 22.3/101.7 MB 26.8 MB/s eta 0:00:03
   ----------- ---------------------------- 28.3/101.7 MB 27.6 MB/s eta 0:00:03
   ------------- -------------------------- 33.8/101.7 MB 27.4 MB/s eta 0:00:03
   --------------- ------------------------ 39.3/101.7 MB 27.4 MB/s eta 0:00:03
   ----------------- ---------------------- 45.4/101.7 MB 27.5 MB/s eta 0:00:03
   -------------------- ------------------- 51.4/101.7 MB 27.8 MB/s eta 0:00:02
   ---------------------- ----------------- 57.1/101.7 MB 28.0 MB/s eta 0:00:02
   ------------------------ --------------- 63.2/101.7 MB 27

In [30]:
def get_score(y_true,y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)    

    print(f"Accuracy: {accuracy:.2f}") 
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1 Score:  {f1:.2f}")

In [ ]:
# file = pd.read_excel("C:\\Users\\Anjali\\Desktop\\AI_Course\\IIT Delhi\\Project\\Capstone Project Problem Statements, Submission Guidelines & Evaluation Procedure\\Credit card fraud detection\\Data.xlsx")
file = pd.read_csv("..\\data\\creditcard.csv")

In [23]:

#split dataset in features and target variable

from sklearn.model_selection import train_test_split


feature_cols = ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10','V11','V12','V13','V14','V15','V16','V17','V18','V19','V20','V21','V22','V23','V24','V25','V26','V27','V28','Amount']  # Example feature columns
label =  ['Class']   #Features used for the example
X = file.drop(columns=['Class'])                            # Features
y = file[label]                                        # Target variable

 # Split dataset into training set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(199364, 30)
(85443, 30)
(199364, 1)
(85443, 1)


In [24]:
try:
    import xgboost as xgb
    print(f"XGBoost is installed! Version: {xgb.__version__}")
except ImportError:
    print("XGBoost is NOT installed.")

XGBoost is installed! Version: 3.2.0


In [25]:
def getXgBoost(n_estimators, max_depth, learning_rate):
    xgb_classifier = xgb.XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate)
    return xgb_classifier

In [27]:
def getXgBoostWithGridSearch(n_estimators, max_depth, learning_rate,scale_pos_weight,use_label_encoder,eval_metric,n_splits,shuffle,random_state,cv,scoring,n_jobs):
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
    param_grid = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'learning_rate': learning_rate,
        'scale_pos_weight': scale_pos_weight # Test standard vs. theoretical balance
    }
    xgb_model = xgb.XGBClassifier(use_label_encoder=use_label_encoder, eval_metric=eval_metric)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs
    )
    return grid_search

In [31]:
xgb_classifier = getXgBoost(100, 3, 0.1)
xgb_classifier.fit(X_train, y_train)
y_pred = xgb_classifier.predict(X_test)
get_score(y_test, y_pred)
cm=confusion_matrix(y_test, y_pred)
print("Confusion matrix \n", cm)


Accuracy: 1.00
Precision: 0.93
Recall:    0.72
F1 Score:  0.81
Confusion matrix 
 [[85301     7]
 [   38    97]]


In [ ]:
## for imbalance data set 
grid_search = getXgBoostWithGridSearch([100, 200], [3, 5], [0.1, 0.2], [1, 2], False, 'logloss', 5, True, 42, None, 'roc_auc', -1)
grid_search.fit(X_train, y_train)
y_pred = grid_search.predict(X_test)
get_score(y_test, y_pred)
cm=confusion_matrix(y_test, y_pred)
print("Confusion matrix \n", cm)

c:\Users\Anjali\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
C:\Users\Anjali\AppData\Local\Temp\ipykernel_20900\3198907507.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
c:\Users\Anjali\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:02:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 1.00
Precision: 0.93
Recall:    0.72
F1 Score:  0.81
Confusion matrix 
 [[85301     7]
 [   38    97]]
